# TP4 – Tests de primalité (partie 1 : test de Fermat)

Décider si un entier $N$ est premier est une opération de base en arithmétique effective : la cryptographie à clé publique a besoin de produire, en quelques millisecondes, des nombres premiers de plusieurs centaines de chiffres.

Essayer tous les diviseurs jusqu'à $\sqrt N$ demande environ $2^{256}$ opérations pour un $N$ de $512$ bits : c'est hors de portée. On va donc construire des tests **probabilistes**, étudier leur coût et, surtout, quantifier leur probabilité d'erreur. Cette première partie est consacrée au test de Fermat ; la seconde partie traitera du test de Miller–Rabin.

**Conventions et fonctions Sage utiles.**

* Les tests probabilistes renvoient `False` si l'entier est *certainement composé* et `True` s'il est *probablement premier*.
* `is_prime(N)` : teste si $N$ est premier, avec une réponse certaine. On s'en sert comme référence pour vérifier nos tests.
* `gcd(a, N)` : plus grand commun diviseur de $a$ et $N$.
* `euler_phi(N)` : indicatrice d'Euler $\varphi(N)$, nombre d'entiers de $[1, N]$ premiers avec $N$.
* `power_mod(a, e, N)` : calcul de $a^{e} \bmod N$ par exponentiation rapide (interdit dans l'exercice 1).
* `primes_first_n(n)` : liste des $n$ premiers nombres premiers.
* `randint(a, b)` : entier aléatoire de l'intervalle $[a, b]$, bornes comprises.
* `factor(N)` : factorisation de $N$ en produit de nombres premiers.
* `len(L)`, `sum(L)` : longueur et somme d'une liste ; on construit commodément des listes en compréhension, par exemple `[a for a in range(1, N) if gcd(a, N) == 1]`.

Les cellules *Réponse :* sont à compléter par un texte rédigé.

## Exercice 1. Exponentiation modulaire rapide

Le test de Fermat repose entièrement sur le calcul de $a^{e} \bmod N$. Avant toute chose, il faut savoir le faire efficacement.

### Exercice 1.a — Exponentiation naïve

La méthode la plus directe pour calculer $a^{e} \bmod N$ consiste à multiplier $a$ par lui-même $e-1$ fois, en réduisant modulo $N$ après chaque multiplication pour que les entiers manipulés restent inférieurs à $N^2$.

Écrire une fonction `puissance_mod_naive(a, e, N)` qui implémente cette méthode et renvoie $a^{e} \bmod N$.

1. Combien de multiplications effectue-t-on, en fonction de $e$ ?
2. Dans un test de primalité, on calcule $a^{N-1} \bmod N$ pour des entiers $N$ de $512$ bits, soit environ $155$ chiffres décimaux. En supposant qu'un ordinateur effectue $10^{9}$ multiplications modulaires par seconde, combien de temps prendrait ce calcul ? Que peut-on en conclure ?

*Réponse :*

### Exercice 1.b — Exponentiation rapide (Square and multiply)

L'idée de la méthode des carrés successifs (en anglais *square-and-multiply*) est d'écrire l'exposant en base $2$. Si
$$e = \sum_{i=0}^{\ell-1} e_i 2^{i}, \qquad e_i \in \{0,1\},$$
alors
$$a^{e} = \prod_{i\,:\,e_i = 1} a^{2^{i}},$$
et les puissances $a^{2^{i}}$ s'obtiennent les unes des autres par un simple carré : $a^{2^{i+1}} = \left(a^{2^{i}}\right)^{2}$.

On parcourt donc les bits de $e$ du moins significatif au plus significatif, en maintenant deux quantités : la puissance courante $a^{2^{i}}$, que l'on élève au carré à chaque étape, et un accumulateur dans lequel on multiplie cette puissance lorsque le bit $e_i$ vaut $1$. Pour $e = 13 = 1101_2$, cela donne
$$a^{13} = a^{8} \cdot a^{4} \cdot a,$$
ce qui nécessite $3$ élévations au carré et $2$ multiplications, contre $12$ multiplications avec la méthode naïve.

Comme à l'exercice 1.a, on réduit modulo $N$ après chaque opération, de sorte qu'aucun entier manipulé n'excède $N^2$.

Écrire une fonction `puissance_mod(a, e, N)` qui calcule $a^{e} \bmod N$ par cette méthode. On s'interdit bien sûr `power_mod` et `pow(a, e, N)`.

*Indication.* Le bit de poids faible de $e$ est `e % 2`, et on passe au bit suivant avec `e // 2`.

1. Combien de multiplications effectue-t-on, en fonction de $e$ ? Donner un encadrement du nombre de multiplications. Pour quels exposants $e$ obtient-on le meilleur cas ? Et le pire cas ?
2. Reprendre la question 2 de l'exercice 1.a avec cette méthode.

*Réponse :*

**Dans toute la suite du TP**, on calcule $a^{e} \bmod N$ avec la fonction `power_mod(a, e, N)` de Sage, qui implémente l'exponentiation rapide.

## Exercice 2. Test de primalité de Fermat

Le test de Fermat repose sur le *petit théorème de Fermat* : Soit $N\in \mathbb Z_{>0}$ et $a$ un entier premier avec $N$. **Si $N$ est premier**, alors
$$a^{N-1} \equiv 1 \pmod N.$$


### Exercice 2.a — Le petit théorème de Fermat

Vérifier expérimentalement que le petit théorème de Fermat est vérifié pour les 100 premiers nombres premiers. 

*Indication.* `primes_first_n`, `power_mod`.

### Exercice 2.b — Pseudo-premiers et nombres de Carmichael

La réciproque du petit théorème de Fermat est fausse. Déterminer le plus petit entier composé impair $N$ tel que $2^{N-1} \equiv 1 \pmod N$ : un tel $N$ est appelé *pseudo-premier de Fermat de base $2$*.

Pire : il existe des entiers composés $N$ tels que $a^{N-1} \equiv 1 \pmod N$ pour **toute** base $a$ première avec $N$. Ces entiers sont les *nombres de Carmichael*. Calculer ceux qui sont $< 10000$. On rappelle que $1$ et les nombres premiers ne sont pas des nombres de Carmichael, car ils ne sont pas composés.

*Indication.* `is_prime`, `gcd`, `power_mod`. Pour tester si $N$ est de Carmichael, on peut s'arrêter dès qu'on trouve une base qui ne vérifie pas la congruence.

### Exercice 2.c — Le test de Fermat

Le test de Fermat repose sur l'idée suivante : si $N$ est composé, il est peu probable que $a^{N-1} \equiv 1 \pmod N$ pour une base $a$ choisie au hasard. Lorsque c'est pourtant le cas, on dit que $N$ est *pseudo-premier de Fermat de base $a$*. On répète donc l'essai avec plusieurs bases aléatoires, selon le pseudo-code suivant.

**Entrées :**
* $N$, l'entier à tester, impair et $\ge 5$ ;
* $k$, le nombre de bases aléatoires à essayer.

**Sortie :**
* `False` si $N$ est certainement composé ;
* `True` si $N$ est probablement premier.

**Algorithme :** pour $i$ de $1$ à $k$,
1. choisir une base $a$ au hasard dans $[2, N-2]$ ;
2. si $\gcd(a, N) \neq 1$, renvoyer `False` ;
3. si $a^{N-1} \not\equiv 1 \pmod N$, renvoyer `False`.

Si les $k$ bases ont passé le test, renvoyer `True`.

*Indication.* `randint`, `gcd`, `power_mod`.

1. Écrire une fonction `test_fermat(N, k)` qui implémente cet algorithme.
2. Que nous apprend le cas $\gcd(a, N) \neq 1$, au-delà de la non-primalité de $N$ ?

*Réponse :*

### Exercice 2.d — Analyse du test

1. Expliquer pourquoi le test donne toujours une réponse correcte lorsque $N$ est premier.
2. Expliquer pourquoi il peut se tromper lorsque $N$ est un nombre de Carmichael, et exprimer en fonction de $\varphi(N)$ la probabilité qu'une itération du test se trompe sur un tel $N$.

*Réponse :*

### Exercice 2.e — Menteurs de Fermat

Pour comprendre d'où vient la probabilité d'erreur, on compte les bases qui trompent le test. On appelle *menteur de Fermat* pour $N$ composé un entier $a \in [1, N-1]$ premier avec $N$ et vérifiant $a^{N-1} \equiv 1 \pmod N$.

1. Écrire une fonction `menteurs_fermat(N)` qui renvoie la liste des menteurs de Fermat pour $N$.
2. Pour chaque entier composé impair $N < 2000$, calculer la proportion de menteurs parmi les entiers de $[1, N-1]$ premiers avec $N$, c'est-à-dire $\#\{\text{menteurs}\} / \varphi(N)$. Séparer les nombres de Carmichael des autres. Quelle majoration semble valoir pour les composés qui ne sont pas de Carmichael ? Est-elle atteinte ? (On démontrera cette majoration en TD.)
3. Que vaut cette proportion pour un nombre de Carmichael ? Qu'en conclure sur le test de Fermat ?

*Indication.* `euler_phi`, `is_prime`, et la liste des nombres de Carmichael obtenue à l'exercice 2.b.

*Réponse :*

### Exercice 2.f — Taux d'erreur du test de Fermat

D'après l'exercice 2.d, le test ne se trompe jamais sur un nombre premier : ses seules erreurs consistent à déclarer « probablement premier » un entier composé. On appelle donc *taux d'erreur* la proportion de ces réponses incorrectes parmi les entiers composés testés. On peut le mesurer de deux façons :
* **sur un échantillon d'entiers** : on applique le test une fois à chaque entier de l'échantillon, et on calcule la proportion d'entiers composés déclarés « probablement premiers » ;
* **sur un entier composé $N$ fixé** : comme le test est aléatoire, on l'applique un grand nombre de fois au même $N$, et on calcule la proportion d'exécutions qui répondent `True`.

1. Pour $k = 4$, estimer le taux d'erreur de `test_fermat` sur un échantillon de $10000$ entiers impairs $N \ge 5$ tirés au hasard sous $10^5$.
2. Pour $k = 4$, estimer le taux d'erreur de `test_fermat` sur chacun des nombres de Carmichael $< 10000$, en effectuant $1000$ exécutions pour chacun.
3. Commenter l'écart entre les deux mesures, et confronter les valeurs obtenues à la question 2 à la formule de l'exercice 2.d.

*Indication.* `is_prime` pour savoir si un entier de l'échantillon est composé ; `euler_phi` pour la formule de l'exercice 2.d.

*Réponse :*